In [ ]:
!pip install mne

     |████████████████████████████████| 6.6MB 2.7MB/s 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [ ]:
def band_pass_filter(eeg, freq_range):
  sfreq = 250
  info = mne.create_info(22, 250, ch_types=["eeg"] * 22)
  raw = mne.io.RawArray(eeg.T, info)
  iir_params = dict(order=5, ftype='cheby2', rs=2.)
  # iir_params.update(ftype='cheby2',rp=1.,  # dB of acceptable pass-band ripple)
  # x = raw._data.T
  # filt = mne.filter.create_filter(x, sfreq, l_freq=freq_range[0], h_freq=freq_range[1],
  #                               method='iir', iir_params=iir_params,
  #                               verbose=True)
  raw = raw.filter(freq_range[0], freq_range[1], fir_design='firwin', method='iir', iir_params=iir_params)

  return raw._data.T
  # return filt

def load_data(mode='train', fno = 1):
  if fno != 4:
    if (mode=='train'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 3+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)

    elif (mode=='test'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 3+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)
      
    return df

  else:
    if (mode=='train'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 1+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)

    elif (mode=='test'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 1+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)
      
    return df

In [ ]:
def drop_classes(df):
  i = 0
  indexes_to_drop = []
  while i < len(df):
    if(df['class'][i]==4):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    elif(df['class'][i]==3):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    else:
      i += 1

  indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
  df_sliced = df.take(list(indexes_to_keep))

  df_sliced = df_sliced.reset_index(drop=True)
  return df_sliced

In [ ]:
def convert_class(sfreq, trigger_points, df):
  asynch_label = []
  i = 0
  while i < len(df['class']):
      if(df['class'][i] == 1):
          end = (sfreq * trigger_points[1]) + 1
          for j in range(i, i+end):
              asynch_label.append(1)
          i += end
      elif(df['class'][i] == 2):
          end = (sfreq * trigger_points[2]) + 1
          for j in range(i, i+end):
              asynch_label.append(2)
          i += end
      elif(df['class'][i] == 3):
          end = (sfreq * trigger_points[3]) + 1
          for j in range(i, i+end):
              asynch_label.append(3)
          i += end
      elif(df['class'][i] == 4):
          end = (sfreq * trigger_points[4]) + 1
          for j in range(i, i+end):
              asynch_label.append(4)
          i += end
      elif(df['class'][i] == 0):
          asynch_label.append(0)
          i += 1

  return np.array(asynch_label)

def prune_records(data_new, label):
  train_data = []
  label_data = []

  for i in range(len(label)):
      if label[i] != 0:
          label_data.append(label[i])
          train_data.append(data_new[i])
  label_data = np.array(label_data)
  train_data = np.array(train_data)

  return train_data, label_data

In [ ]:
def data_win(sfreq, data, asynch_label):
  sampling_window = 2 * sfreq
  shift_length = 1 * sfreq
  t_start = 0
    
  new_data = []
  labels = []


  while t_start + sampling_window < data.shape[0]:
      new_data.append(data[t_start:t_start+sampling_window, :].T)
      labels.append(asynch_label[t_start:t_start+sampling_window])
    
      t_start = t_start + shift_length
    
  return np.array(new_data), np.array(labels)

def transform_label(label_new):
  label = []

  for i in label_new:
      count1 = np.count_nonzero(i==1)
      count2 = np.count_nonzero(i==2)
      count3 = np.count_nonzero(i==3)
      count4 = np.count_nonzero(i==4)
      if count1 >= 250:
          to_add = 1
      elif count2 >= 250:
          to_add = 2
      elif count3 >= 250:
          to_add = 3
      elif count4 >= 250:
          to_add = 4
      else:
          to_add = 0
      label.append(to_add)
      
  label = np.array(label)

  return label

In [ ]:
def fbcsp(df, labels, sfreq, train=True, csp_objects=None):
  freq = 4
  increment = 4
  end_freq = 40
  if train==True:
    csp_objects = []
    csp_data = []
    while freq < end_freq:
      freq_range = []
      freq_range.append(freq)
      freq_range.append(freq+increment)
      freq += increment
      out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

      X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
      y = transform_label(y)
      X, y = prune_records(X, y)

      csp = CSP(n_components=2, reg=None, log=True, norm_trace=False)
      final_data = csp.fit_transform(X, y)

      csp_objects.append(csp)
      csp_data.append(final_data)

    return np.array(csp_objects), np.array(csp_data), y
  
  else:
    csp_data = []
    count = 0
    while freq < end_freq:
      freq_range = []
      freq_range.append(freq)
      freq_range.append(freq+increment)
      freq += increment
      out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

      X, y = data_win(sfreq=sfreq, data=out_data, asynch_label=labels)
      y = transform_label(y)
      X, y = prune_records(X, y)

      final_data = csp_objects[count].transform(X)
      count += 1

      csp_data.append(final_data)

    return np.array(csp_data), y

  # return np.array(csp_objects), np.array(csp_data), y


In [ ]:
def itr(n_class, p_class, c_time):
  B = (np.log2(n_class) + (p_class * np.log2(p_class)) + ((1-p_class) * np.log2((1-p_class)/(n_class-1)))) / c_time * 60

  return B

def performance_metrics(y_test, y_pred):
  acc = accuracy_score(y_test, y_pred)
  print('Accuracy Score: ', acc)
  print('Cohen Kappa Score: ', cohen_kappa_score(y_test, y_pred))
  print('ITR (bits per minute): ', itr(n_class=2, p_class=acc, c_time=2))
  print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))

In [ ]:
def get_train_data(fno):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}

  df = load_data(mode='train', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)
  asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)
  csp_objects, csp_data, y = fbcsp(train_df, asynch_label, sfreq, train=True)

  final_data = pd.DataFrame(csp_data[0])
  col_count = 4
  for i in range(1, len(csp_data)):
    for j in range(len(csp_data[i].T)):
      final_data[str(col_count)] = csp_data[i].T[j]
      col_count += 1

  return final_data.values, y, csp_objects

def get_eval_data(fno, csp_objects):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}

  df = load_data(mode='test', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)
  asynch_label = convert_class(sfreq=sfreq, trigger_points=trigger_points, df=train_df)
  csp_data, y = fbcsp(train_df, asynch_label, sfreq, train=False, csp_objects=csp_objects)

  final_data = pd.DataFrame(csp_data[0])
  col_count = 4
  for i in range(1, len(csp_data)):
    for j in range(len(csp_data[i].T)):
      final_data[str(col_count)] = csp_data[i].T[j]
      col_count += 1

  return final_data.values, y

In [ ]:
def train_mibif(X, y):
  print('------Train---------')
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

  #get the best k features base on MIBIF algorithm
  select_K = SelectKBest(mutual_info_classif,k=8).fit(X, y)
  extra = select_K.get_support()
  if extra[0] == True:
    extra[1] = True
  for i in range(2, len(extra)):
    if extra[i] == True:
      if i%2 == 0:
        extra[i+1] = True
      else:
        extra[i-1] = True
  pos = np.where(extra==False)
  New_train = np.delete(X_train, list(pos[0]), 1)
  New_test = np.delete(X_test, list(pos[0]), 1)
  # New_train=select_K.transform(X_train)
  # New_test=select_K.transform(X_test)
  ss = StandardScaler()
  New_train = ss.fit_transform(New_train,y_train)
  New_test = ss.transform(New_test)

  print('####### SVM#####')
  svm = SVC()
  svm.fit(New_train, y_train)
  y_pred = svm.predict(New_test)
  performance_metrics(y_test, y_pred)
  
  print('##########LDA#########')
  lda = LinearDiscriminantAnalysis()
  lda.fit(New_train, y_train)
  y_pred = lda.predict(New_test)
  performance_metrics(y_test, y_pred)

  return list(pos[0]), svm, lda, ss

def eval_mibif(pos, svm, lda, ss, X, y):
  print('------Test--------')
  X = np.delete(X, pos, 1)
  X = ss.transform(X)
  print('#####SVM######')
  y_pred = svm.predict(X)
  performance_metrics(y, y_pred)
  print('#####LDA######')
  y_pred = lda.predict(X)
  performance_metrics(y, y_pred)


In [ ]:
def train_rf(X, y):
  print('--------Train--------')
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
  clf = RandomForestClassifier(random_state=42)
  ss = StandardScaler()
  X_train = ss.fit_transform(X_train, y_train)
  clf.fit(X_train, y_train)
  X_test = ss.transform(X_test)
  y_pred = clf.predict(X_test)
  ####Random Forest#######
  performance_metrics(y_test, y_pred)

  return clf, ss
def eval_rf(clf, ss, X, y):
  print('--------Test------')
  X = ss.transform(X)
  y_pred = clf.predict(X)
  performance_metrics(y, y_pred)

# Subjects A1

In [ ]:
X, y, csp_objects = get_train_data(fno=1)
X_eval, y_eval = get_eval_data(fno=1,csp_objects=csp_objects)

In [ ]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.7862068965517242
Cohen Kappa Score:  0.572189968592367
ITR (bits per minute):  7.539875003712037
Confusion Matrix:  [[54 18]
 [13 60]]
##########LDA#########
Accuracy Score:  0.7862068965517242
Cohen Kappa Score:  0.5722713864306784
ITR (bits per minute):  7.539875003712037
Confusion Matrix:  [[55 17]
 [14 59]]
------Test--------
#####SVM######
Accuracy Score:  0.6701388888888888
Cohen Kappa Score:  0.3402777777777778
ITR (bits per minute):  2.556467767582663
Confusion Matrix:  [[194  94]
 [ 96 192]]
#####LDA######
Accuracy Score:  0.6440972222222222
Cohen Kappa Score:  0.2881944444444444
ITR (bits per minute):  1.8231137653209661
Confusion Matrix:  [[203  85]
 [120 168]]


In [ ]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.7586206896551724
Cohen Kappa Score:  0.5169886742171885
ITR (bits per minute):  6.080204695576733
Confusion Matrix:  [[52 20]
 [15 58]]
--------Test------
Accuracy Score:  0.6579861111111112
Cohen Kappa Score:  0.3159722222222222
ITR (bits per minute):  2.1980150035926025
Confusion Matrix:  [[194  94]
 [103 185]]


# Subjects 2-3 and 5-6

In [ ]:
X, y, csp_objects = get_train_data(fno=2)
X_eval, y_eval = get_eval_data(fno=2,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 40 (2.2e-16 eps * 22 dim * 8.3e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 39 (2.2e-16 eps * 22 dim * 8e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using

In [ ]:
X, y, csp_objects = get_train_data(fno=3)
X_eval, y_eval = get_eval_data(fno=3,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 44 (2.2e-16 eps * 22 dim * 8.9e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 45 (2.2e-16 eps * 22 dim * 9.2e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi

In [ ]:
X, y, csp_objects = get_train_data(fno=4)
X_eval, y_eval = get_eval_data(fno=4,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 41 (2.2e-16 eps * 22 dim * 8.3e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 41 (2.2e-16 eps * 22 dim * 8.4e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi

In [ ]:
X, y, csp_objects = get_train_data(fno=5)
X_eval, y_eval = get_eval_data(fno=5,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 43 (2.2e-16 eps * 22 dim * 8.8e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 43 (2.2e-16 eps * 22 dim * 8.9e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi

In [ ]:
X, y, csp_objects = get_train_data(fno=6)
X_eval, y_eval = get_eval_data(fno=6,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 59 (2.2e-16 eps * 22 dim * 1.2e+16  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 58 (2.2e-16 eps * 22 dim * 1.2e+16  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi

In [ ]:
X, y, csp_objects = get_train_data(fno=7)
X_eval, y_eval = get_eval_data(fno=7,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 39 (2.2e-16 eps * 22 dim * 7.9e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 40 (2.2e-16 eps * 22 dim * 8.2e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi

In [ ]:
X, y, csp_objects = get_train_data(fno=8)
X_eval, y_eval = get_eval_data(fno=8,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 52 (2.2e-16 eps * 22 dim * 1.1e+16  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 51 (2.2e-16 eps * 22 dim * 1.1e+16  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi

In [ ]:
X, y, csp_objects = get_train_data(fno=9)
X_eval, y_eval = get_eval_data(fno=9,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Computing rank from data with rank=None
    Using tolerance 60 (2.2e-16 eps * 22 dim * 1.2e+16  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 57 (2.2e-16 eps * 22 dim * 1.2e+16  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance usi